# Calculadora de metas nutricionais diárias

O código nesse *notebook* puxa informações nutricionais de múltiplas fontes para cálculo da quantidade de elementos nutricionais diários listados pela TACO para uma alimentação saudável.

- Fontes:
    - UNICAMP/NEPA - Tabela Brasileira de Composição de Alimentos (TACO);
    - NASEM/Health Canada - Dietary Reference Intakes, equations to estimate energy requirement;
    - Health Canada / Food and Nutrition Board - Dietary Reference Intakes tables;
    - National Academies 2019 - Dietary Reference Intakes for Sodium and Potassium;
    - WHO/FAO/UNU 2007 and FAO 2011 protein quality consultation tables.

In [1]:
import csv
from pathlib import Path
import pandas as pd

from functions import (
    calcular_necessidades,
    formatar_numero_brasileiro,
    formatar_numero_exportacao,
    limpar_rotulo,
)

In [ ]:
DATA_DIR = Path("../data")
ID_COL = "Número do Alimento"

alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")

aminoacidos["Triptofano (g)"] = pd.to_numeric(
    aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

taco_completo = alimentos.merge(
    acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

output_path = DATA_DIR / "taco_completo.csv"
taco_completo.to_csv(output_path, index=False, na_rep="NA")

taco_completo.head()

try:
    tabela_bruta = taco_completo.copy()
except NameError:
    alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
    acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
    aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")
    aminoacidos["Triptofano (g)"] = pd.to_numeric(
        aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    tabela_bruta = alimentos.merge(
        acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
    ).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

colunas_repetidas = [
    coluna
    for coluna in tabela_bruta.columns
    if coluna.endswith(("_acidos_graxos", "_aminoacidos"))
    and coluna.startswith(("Categoria do alimento", "Descrição dos alimentos"))
]
tabela_limpa = tabela_bruta.drop(columns=colunas_repetidas)

tabela_formatada = tabela_limpa.copy()
colunas_numericas = tabela_formatada.select_dtypes(include="number").columns

for coluna in colunas_numericas:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(formatar_numero_brasileiro)

for coluna in tabela_formatada.columns.difference(colunas_numericas):
    tabela_formatada[coluna] = tabela_formatada[coluna].replace("NA", pd.NA).fillna("")

tabela_formatada.columns = [
    limpar_rotulo(coluna) for coluna in tabela_formatada.columns
]

output_path = DATA_DIR / "taco_completo.csv"
tabela_formatada.to_csv(output_path, sep=";", index=False, quoting=csv.QUOTE_ALL)

tabela_formatada.head()

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Umidade,Energia (kcal),Energia (kJ),Proteína (g),Lipídeos (g),Colesterol (mg),Carboidrato (g),...,Tirosina (g),Valina (g),Arginina (g),Histidina (g),Alanina (g),Ácido Aspártico (g),Ácido Glutâmico (g),Glicina (g),Prolina (g),Serina (g)
0,1,Cereais e derivados,"Arroz, integral, cozido","70,1",124,517,"2,6",1,,"25,8",...,,,,,,,,,,
1,2,Cereais e derivados,"Arroz, integral, cru","12,2",360,1505,"7,3","1,9",,"77,5",...,,,,,,,,,,
2,3,Cereais e derivados,"Arroz, tipo 1, cozido","69,1",128,537,"2,5","0,2",,"28,1",...,,,,,,,,,,
3,4,Cereais e derivados,"Arroz, tipo 1, cru","13,2",358,1497,"7,2","0,3",,"78,8",...,,,,,,,,,,
4,5,Cereais e derivados,"Arroz, tipo 2, cozido","68,7",130,544,"2,6","0,4",,"28,2",...,,,,,,,,,,


In [ ]:
necessidades_nutricionais = calcular_necessidades(
    sexo="masculino",
    idade_anos=25,
    altura_m=1.84,
    peso_kg=136,
    colunas_taco=list(tabela_formatada.columns)
)

necessidades_exportacao = necessidades_nutricionais.copy()
for coluna in ["EER usado (kcal/dia)", "Alvo", "Mínimo", "Máximo"]:
    necessidades_exportacao[coluna] = necessidades_exportacao[coluna].map(
        formatar_numero_exportacao
    )

saida_necessidades = Path("../data") / "necessidades_nutricionais_estimadas.csv"
necessidades_exportacao.to_csv(
    saida_necessidades, sep=";", index=False, quoting=csv.QUOTE_ALL
)

necessidades_nutricionais

,Estágio de vida,EER usado (kcal/dia),Nutriente,Colunas TACO usadas,Tipo,Alvo,Mínimo,Máximo,Unidade,Base científica,Observações
0,male_19_30,4098,Energia,Energia (kcal),EER,4098.0,NaN,NaN,kcal/dia,"Equação NASEM 2023 por sexo, idade, altura, pe...",
1,male_19_30,4098,Carboidrato,Carboidrato (g),RDA + AMDR,130.0,461.0,665.8,g/dia,DRI: RDA e 45-65% da energia,
2,male_19_30,4098,Proteína,Proteína (g),RDA por kg + AMDR,108.8,102.4,358.5,g/dia,"0.8 g/kg/dia, com mínimo de referência 56 g/dia",
3,male_19_30,4098,Lipídeos totais,Lipídeos (g),AMDR,NaN,91.1,159.3,g/dia,Percentual de energia vindo de gorduras totais,
4,male_19_30,4098,Fibra Alimentar,Fibra Alimentar (g),AI estimada por energia,57.4,NaN,NaN,g/dia,14 g/1000 kcal; tabela DRI também informa AI p...,AI do estágio de vida na tabela: 38 g/dia
...,...,...,...,...,...,...,...,...,...,...,...
56,male_19_30,4098,Ácido Aspártico (g),Ácido Aspártico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
57,male_19_30,4098,Ácido Glutâmico (g),Ácido Glutâmico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
58,male_19_30,4098,Glicina (g),Glicina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
59,male_19_30,4098,Prolina (g),Prolina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
